# PocketLFM — train on Kaggle (all-in-one)

**Before running:** open the right-hand panel → *Session options* → set **Accelerator = GPU T4 x2** and **Internet = On**.

Then **Run All**. The notebook will, in order:
1. Clone the repo + install it.
2. Preprocess LJSpeech → Mimi latents (only the first time; reused after).
3. Train with auto-resume + best-checkpoint saving.
4. Synthesize a sample from the best checkpoint.

**To continue past the 12h limit:** *+ Add Input* → *Notebook Output* → add **this notebook's previous version**, then **Save & Run All (Commit)** again. It auto-detects the cached latents and the last checkpoint and resumes.

In [ ]:
REPO_URL   = "https://github.com/emmudsouza/poclfmtts.git"
EPOCHS     = 300
BATCH_SIZE = 32
GRAD_ACCUM = 1

import os
os.chdir("/kaggle/working")
if os.path.isdir("poclfmtts"):
    !cd poclfmtts && git pull
else:
    !git clone {REPO_URL}
%cd /kaggle/working/poclfmtts
!pip install -q -e . --no-deps && pip install -q pocket-tts
print("setup done")

In [ ]:
# Reuse a COMPLETE latent cache if available; otherwise (re)build it fresh.
import os, glob, shutil
MIN_CLIPS = 12000  # LJSpeech has ~13100; a smaller cache means a broken/partial build

def cache_count(c):
    idx = os.path.join(c, "index.txt")
    return len([x for x in open(idx).read().split() if x]) if os.path.exists(idx) else 0

candidates = glob.glob("/kaggle/input/*/ljspeech_cache") + ["/kaggle/working/ljspeech_cache"]
CACHE = next((c for c in candidates if cache_count(c) >= MIN_CLIPS), None)
if CACHE is None:
    CACHE = "/kaggle/working/ljspeech_cache"
    shutil.rmtree(CACHE, ignore_errors=True)   # drop any partial cache
    shutil.rmtree("data", ignore_errors=True)  # drop any partial download
    print("Building latent cache (one-time, ~30-40 min)...")
    !python scripts/data_ljspeech.py --download --out {CACHE} --device cuda
else:
    print(f"Reusing cache: {CACHE} ({cache_count(CACHE)} clips)")

In [ ]:
# Resume from a previous session's checkpoint if one is mounted as input.
import glob, shutil, os
OUT = "/kaggle/working/runs/exp1"
os.makedirs(OUT, exist_ok=True)
prev = glob.glob("/kaggle/input/*/pocketlfm_last.pt") + glob.glob("/kaggle/input/*/runs/exp1/pocketlfm_last.pt")
if prev:
    shutil.copy(prev[0], os.path.join(OUT, "pocketlfm_last.pt"))
    print("Resuming from", prev[0])

!python scripts/train.py --cache {CACHE} --out {OUT} --epochs {EPOCHS} \
    --batch-size {BATCH_SIZE} --grad-accum {GRAD_ACCUM} --val-split 0.02 --workers 2 --resume auto

# Persist checkpoints to the notebook output so the next session can resume.
for f in ("pocketlfm_best.pt", "pocketlfm_last.pt"):
    p = os.path.join(OUT, f)
    if os.path.exists(p):
        shutil.copy(p, "/kaggle/working/" + f)
print("training cell done")

In [ ]:
# Synthesize a sample from the best checkpoint.
import os, IPython.display as ipd
best = "/kaggle/working/runs/exp1/pocketlfm_best.pt"
if os.path.exists(best):
    !python scripts/infer.py --ckpt {best} --text "hello from the cloud" \
        --out /kaggle/working/sample.wav --device cuda
    ipd.display(ipd.Audio("/kaggle/working/sample.wav"))
else:
    print("No checkpoint yet — let training run longer.")